<a href="https://colab.research.google.com/github/HiveCase/DA4001-AI/blob/main/Week3/GA_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import random

## Question 1: Create a 3x5 tensor with random floating-point values between 0 and 1 after setting the seed to 42 using the torch library. Compute the sum of all elements in the tensor. (write the answer correct upto 2 decimals)


In [2]:
torch.manual_seed(42)
x = torch.rand(3,5)
x

tensor([[0.8823, 0.9150, 0.3829, 0.9593, 0.3904],
        [0.6009, 0.2566, 0.7936, 0.9408, 0.1332],
        [0.9346, 0.5936, 0.8694, 0.5677, 0.7411]])

In [3]:
torch.sum(x)

tensor(9.9613)

In [4]:
print(f"The sum of all elements in the tensor is: {round((torch.sum(x).item()),2)}")

The sum of all elements in the tensor is: 9.96


## Question 2: what is the value of `q3(z)` in given below equation? where `z = 1.57` (write the answer correct upto 2 decimals)

$$
q_1(z) = z^2
$$
$$
q_2(z) = z^3
$$
$$
q_3(z) = e^z * \sin(z)
$$
$$
p(z) = \frac{q_1}{q_2} + q_3
$$

In [5]:
z = torch.tensor(1.57, requires_grad=True)
p = (z**2/z**3) + (torch.exp(z)*torch.sin(z))
p

tensor(5.4436, grad_fn=<AddBackward0>)

In [6]:
q1 = z**2
q2 = z**3
q3 = (torch.exp(z)*torch.sin(z))
print(f"The value of q3 is: {round(q3.item(),2)}")

The value of q3 is: 4.81


## Question 3: what is the value of $$ \frac{∂p}{∂z}$$ where $$ z = 1.57 $$ (write the answer correct upto 2 decimals)

In [7]:
p.backward()
grad_p = z.grad
print(f"The required gradient is: {round(grad_p.item(),2)}")

The required gradient is: 4.4


## Question 4: what is the value of $$ \frac{∂p}{∂q_3}$$ where $$ z = 1.57 $$ (write the answer correct upto 2 decimals)

In [8]:
p1 = (q1/q2) + q3
q3.retain_grad()
p1.backward()
grad_q3 = q3.grad
print(f"The required gradient is: {round(grad_q3.item(),2)}")

The required gradient is: 1.0


## Question 5: Read the mnist_train_small.csv dataset provided by colab in sample_data folder. First column is the target column so accordingly divide the dataset into features and labels. Before training scale the features using min-max-scaler
$$ X_{scaled} = \frac{X - min}{max-min}$$

-  Use `dtype = torch.float32` for feature tensor
-  Use `dtype = torch.long` for label tensor
-  Build the following network
     - Assume the input and output layer according to the data
     - first hidden layer with 128 neurons and Relu activation function
     - second hidden layer with 64 neurons and Relu activation function
- Initialize the model with torch.manual_seed(42)
- Use nn.CrossEntropyLoss() for loss calculation (this pytorch function internally applies softmax activation function before calculating the loss)
- Optimize the model using ADAM optimizer with learning_rate = 0.01 and train for 10 epochs


Hint- train the model multiple times if it's showing different initial loss values that means manual_seed(42) not set correctly
Enter the number of parameters (weights & biases both) in the network.

In [9]:
import pandas as pd
import numpy as np
df = pd.read_csv('/content/sample_data/mnist_train_small.csv')
display(df.sample(5))
display(df.shape)

,6,0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,...,0.581,0.582,0.583,0.584,0.585,0.586,0.587,0.588,0.589,0.590
17822,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3968,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10872,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1535,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3872,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


(19999, 785)

In [11]:
from sklearn.preprocessing import MinMaxScaler
# Split features and labels
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

# Scale features
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

# Convert to tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [12]:
import torch.nn as nn

torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
)

print(sum(p.numel() for p in model.parameters()))

109386


## Question 6: What is the difference between the losses computed at last epoch and first epoch ? (correct answer upto two decimals)

In [17]:
import torch.optim as optim
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

losses = []

for epoch in range(10):
    optimizer.zero_grad()

    outputs = model(X)
    loss = criterion(outputs, y)

    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    print(f"Epoch {epoch+1}: {loss.item():.4f}")

print("Difference (first - last):", round(losses[0] - losses[-1], 2))
print("Difference (last - first):", round(losses[-1] - losses[0], 2))

Epoch 1: 2.3140
Epoch 2: 2.1494
Epoch 3: 1.8079
Epoch 4: 1.4198
Epoch 5: 1.0880
Epoch 6: 0.8691
Epoch 7: 0.7179
Epoch 8: 0.6810
Epoch 9: 0.5547
Epoch 10: 0.5733
Difference (first - last): 1.74
Difference (last - first): -1.74


## Question 7: Design a CNN with one convolutional layer (`16 filters, 5x5 kernel, stride = 2, padding = 1`) followed by an average-pooling layer (2x2).

Use the below code to generate the random images

```

img = torch.randint(0, 256, size=(128, 3, 32, 32), dtype=torch.float32)

```

## what will be the output size after the average-pooling layer?

In [19]:
img = torch.randint(0, 256, size=(128, 3, 32, 32), dtype=torch.float32)
conv_layer = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, stride=2, padding=1)
pool = nn.AvgPool2d(kernel_size=2)
out = pool(conv_layer(img))
print(out.shape)

torch.Size([128, 16, 7, 7])


## Question 8: Using the torchvision.datasets module, load the CIFAR-10 training dataset and use torchvision.transforms to convert them into tensors.
## Use DataLoader to load the dataset with the following settings:
```
Batch size = 128
Shuffle = True
```
## How many training data points are there in the last batch?

In [ ]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

transform = transforms.ToTensor()

trainset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

trainloader = DataLoader(
    trainset,
    batch_size=128,
    shuffle=True
)

last_batch = None
for images, labels in trainloader:
    last_batch = images

print(last_batch.shape[0])

  4%|▍         | 6.82M/170M [06:24<4:14:36, 10.7kB/s]